In [1]:
import warnings
import numpy as np
# import lightgbm as lgb
# from fontTools.misc.cython import returns
# from pyarrow.types import is_large_binary
# from sympy.codegen.ast import continue_
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.svm import SVR
# from sklearn.neural_network import MLPRegressor
# from sklearn.tree import DecisionTreeRegressor
# from statsmodels.tools.eval_measures import rmse, hqic_sigma
import pandas as pd
import re
# from optimize_params import *
import matplotlib.pyplot as plt
from datetime import datetime

# np.random.seed(42)

warnings.filterwarnings("ignore", category=RuntimeWarning)

def mape(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE)

    Parameters:
        y_true (array-like): Actual values
        y_pred (array-like): Predicted values

    Returns:
        float: MAPE in percentage (%)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [2]:
df = pd.read_parquet("data/silver/dataset_with_weather_time_features.parquet")

In [3]:
train = df[df['datetime'] < '2025-07-01']
val = df[(df['datetime'] > '2025-06-30') & (df['datetime'] < '2025-08-01')]
test = df[df['datetime'] > '2025-07-31']

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

In [4]:
y_col = 'Sum of кВт'

In [5]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.6.0.dev20241112+cu121
12.1
True


In [6]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from autogluon.core.metrics import make_scorer
from sklearn.metrics import mean_squared_error


def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))


smape_scorer = make_scorer(
    name="SMAPE",
    score_func=smape,
    optimum=0,
    greater_is_better=False
)

# Optional but strongly recommended:
# keep a smaller subset while debugging
# train = train.sample(1_000_000, random_state=42).reset_index(drop=True)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_safe"
)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_gpu",  # new path to avoid any cached state
    verbosity=3,                  # will log "Fitting X with num_gpus: 1"
)

predictor.fit(
    train_data=train,
    tuning_data=val,
    presets="best_quality",
    num_gpus=1,
    dynamic_stacking=False,
    num_bag_folds=0,
    num_stack_levels=0,
    # time_limit=5 * 60,
    # hyperparameters=hp,
    ag_args_fit={
        "ag.max_memory_usage_ratio": 2.0,  # allow up to 2x estimated memory
    },
)

lb = predictor.leaderboard(val, silent=True)
print(lb)

val_pred = predictor.predict(val.drop(columns=[y_col]))
test_pred = predictor.predict(test.drop(columns=[y_col]))

print("\nValidation metrics")
print("SMAPE:", smape(val[y_col], val_pred))
print("RMSE :", rmse(val[y_col], val_pred))
print("MAPE :", mape(val[y_col], val_pred))

print("\nTest metrics")
print("SMAPE:", smape(test[y_col], test_pred))
print("RMSE :", rmse(test[y_col], test_pred))
print("MAPE :", mape(test[y_col], test_pred))

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
GPU Count:          1
Memory Avail:       13.19 GB / 31.11 GB (42.4%)
Disk Space Avail:   30.46 GB / 475.82 GB (6.4%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'num_bag_folds': 0,
 'num_bag_sets': 1,
 'num_stack_levels': 0}
Full kwargs:
{'_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'calibrate': 'auto',
 'ds_args': {'clean_up_fits': True,
             'detection_time_frac': 0.25,
             'enable_ray_logging': True,
             'holdout_data': None,
             'holdout_frac': 

0:	learn: 13.6872884	test: 17.4963647	best: 17.4963647 (0)	total: 352ms	remaining: 352ms
1:	learn: 13.2824344	test: 17.0653217	best: 17.0653217 (1)	total: 373ms	remaining: 0us
bestTest = 17.06532174
bestIteration = 1
0:	learn: 13.6872884	test: 17.4963647	best: 17.4963647 (0)	total: 209ms	remaining: 27.4s
20:	learn: 9.0674975	test: 12.8278362	best: 12.8278362 (20)	total: 672ms	remaining: 3.55s
40:	learn: 7.7411988	test: 11.2438349	best: 11.2438349 (40)	total: 1.11s	remaining: 2.46s
60:	learn: 7.2423914	test: 10.5231216	best: 10.5231216 (60)	total: 1.54s	remaining: 1.79s
80:	learn: 6.9787341	test: 10.1558789	best: 10.1558789 (80)	total: 1.96s	remaining: 1.24s
100:	learn: 6.7665187	test: 9.8764595	best: 9.8764595 (100)	total: 2.38s	remaining: 732ms
120:	learn: 6.6289127	test: 9.6581486	best: 9.6581486 (120)	total: 2.81s	remaining: 255ms
131:	learn: 6.5636941	test: 9.5707566	best: 9.5707566 (131)	total: 3.04s	remaining: 0us
bestTest = 9.570756568
bestIteration = 131


Saving models/autogluon_gpu\models\CatBoost\model.pkl
Saving models/autogluon_gpu\utils\attr\CatBoost\y_pred_proba_val.pkl
	-0.2268	 = Validation score   (-SMAPE)
	26.47s	 = Training   runtime
	0.05s	 = Validation runtime
	6078045.1	 = Inference  throughput (rows/s | 304499 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: ExtraTreesMSE ... Training model for up to 692.19s of the 692.16s of remaining time.
	Fitting ExtraTreesMSE with 'num_gpus': 1, 'num_cpus': 12
Saving models/autogluon_gpu\models\ExtraTreesMSE\model.pkl
Saving models/autogluon_gpu\utils\attr\ExtraTreesMSE\y_pred_proba_val.pkl
	-0.173	 = Validation score   (-SMAPE)
	465.44s	 = Training   runtime
	0.21s	 = Validation runtime
	1453657.2	 = Inference  throughput (rows/s | 304499 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: NeuralNetFastAI ... Training model for up to 226.44s of the 226.42s of remaining time.
	Fitting NeuralNetFastAI with 'num_gpus': 1, 'num_cpus': 6
Fit

[0]	validation_0-rmse:17.10758	validation_0-custom_metric:0.46936
[50]	validation_0-rmse:10.20612	validation_0-custom_metric:0.26951
[100]	validation_0-rmse:9.70983	validation_0-custom_metric:0.24357
[150]	validation_0-rmse:9.46818	validation_0-custom_metric:0.22940
[200]	validation_0-rmse:9.39523	validation_0-custom_metric:0.21929
[250]	validation_0-rmse:9.33416	validation_0-custom_metric:0.21404
[300]	validation_0-rmse:9.34678	validation_0-custom_metric:0.20835
[350]	validation_0-rmse:9.27273	validation_0-custom_metric:0.20441
[400]	validation_0-rmse:9.27558	validation_0-custom_metric:0.20085
[450]	validation_0-rmse:9.20054	validation_0-custom_metric:0.19773
[500]	validation_0-rmse:9.16964	validation_0-custom_metric:0.19556
[550]	validation_0-rmse:9.16765	validation_0-custom_metric:0.19350
[600]	validation_0-rmse:9.16625	validation_0-custom_metric:0.19127
[650]	validation_0-rmse:9.15682	validation_0-custom_metric:0.18928
[700]	validation_0-rmse:9.17686	validation_0-custom_metric:0.18

C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and w

                 model  score_test  score_val eval_metric  pred_time_test  \
0  WeightedEnsemble_L2   -0.164569  -0.164569       SMAPE        3.237563   
1      RandomForestMSE   -0.170742  -0.170742       SMAPE        0.338526   
2        ExtraTreesMSE   -0.172960  -0.172960       SMAPE        0.267237   
3              XGBoost   -0.177320  -0.177320       SMAPE        2.615787   
4             CatBoost   -0.226772  -0.226772       SMAPE        0.053422   
5       KNeighborsDist   -0.491544  -0.491544       SMAPE     1035.159333   
6       KNeighborsUnif   -0.491544  -0.491544       SMAPE     1036.297795   

   pred_time_val     fit_time  pred_time_test_marginal  \
0       2.374431  1594.421385                 0.016013   
1       0.204114  1055.453131                 0.338526   
2       0.209471   465.436297                 0.267237   
3       1.953846    72.867828                 2.615787   
4       0.050098    26.468973                 0.053422   
5     878.757720     4.599724      

Loading: models/autogluon_gpu\models\ExtraTreesMSE\model.pkl
Loading: models/autogluon_gpu\models\RandomForestMSE\model.pkl
Loading: models/autogluon_gpu\models\XGBoost\model.pkl
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warn


Validation metrics
SMAPE: 0.16456865935508957
RMSE : 9.030861185146096
MAPE : 491807.5687340839

Test metrics
SMAPE: 0.18190097998138052
RMSE : 10.068459438746995
MAPE : 0.34638148382408007


In [7]:
importance = predictor.feature_importance(train)
importance

These features in provided data are not utilized by the predictor and will be ignored: ['ОСР опис']
Loading: models/autogluon_gpu\models\WeightedEnsemble_L2\model.pkl
Computing feature importance via permutation shuffling for 36 features using 5000 rows with 5 shuffle sets...
Loading: models/autogluon_gpu\models\ExtraTreesMSE\model.pkl
Loading: models/autogluon_gpu\models\RandomForestMSE\model.pkl
Loading: models/autogluon_gpu\models\XGBoost\model.pkl
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Lev\Miniconda3\envs\beets\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was ren

AmbiguousTimeError: Cannot infer dst time from 2024-10-27 03:00:00, try using the 'ambiguous' argument